# B2-023-generative-models-diffusion — Practice p08 — Solution

**Type:** constrained-coding · **Difficulty:** core · **Concepts:** generative-adversarial-network

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Supplied cell (copied verbatim from the statement)

In [ ]:
import math
import torch

SEED = 20261015
CENTERS = torch.tensor([[1.0, 1.0], [-1.0, 1.0], [-1.0, -1.0], [1.0, -1.0]])
S = torch.tensor([[1.1, 0.9], [1.5, 1.0], [-1.0, 1.4], [-1.2, -1.2], [0.0, 0.0],
                  [1.0, -1.6], [-0.6, 1.0], [0.9, -1.2], [3.0, 3.0], [-1.0, 0.5]])
S3 = torch.cat([CENTERS[0].repeat(20, 1), CENTERS[1].repeat(2, 1), CENTERS[2].repeat(1, 1),
                torch.tensor([[5.0, 5.0]]).repeat(41, 1)])

## Solution

The rule is implemented with one broadcast squared-distance matrix of shape `(n,K)`.
`argmin(dim=1)` returns the **first** minimizer, so exact ties go to the lowest index.
A sample counts only when its squared distance to its assigned centre is `<= radius ** 2`, and `torch.bincount(..., minlength=K)` gives an int64 count vector of shape `(K,)`.
A component is covered when its count reaches `math.ceil(n / (8 * K))`.

In [ ]:
def _check(samples, centers, radius):
    if not isinstance(samples, torch.Tensor) or not isinstance(centers, torch.Tensor):
        raise ValueError("samples and centers must be tensors")
    if samples.dim() != 2 or centers.dim() != 2:
        raise ValueError("samples and centers must be 2-D")
    if samples.shape[1] != centers.shape[1]:
        raise ValueError("samples and centers must share their last dimension")
    if samples.shape[0] == 0:
        raise ValueError("need at least one sample")
    if not isinstance(radius, (int, float)) or isinstance(radius, bool) or not radius > 0:
        raise ValueError("radius must be a positive number")


def _counts(samples, centers, radius, strict=False):
    d2 = ((samples[:, None, :] - centers[None, :, :]) ** 2).sum(dim=2)     # (n, K)
    nearest = d2.argmin(dim=1)                                              # ties -> lowest index
    nearest_d2 = d2.gather(1, nearest[:, None]).squeeze(1)
    inside = nearest_d2 < radius ** 2 if strict else nearest_d2 <= radius ** 2
    return torch.bincount(nearest[inside], minlength=centers.shape[0]).to(torch.int64)


def mode_coverage(samples, centers, radius):
    _check(samples, centers, radius)
    counts = _counts(samples, centers, radius)
    need = math.ceil(samples.shape[0] / (8 * centers.shape[0]))
    covered = int((counts >= need).sum().item())
    return counts, covered


def mode_coverage_strict(samples, centers, radius):
    """Defective local variant for probe 2: '<' instead of '<='."""
    _check(samples, centers, radius)
    counts = _counts(samples, centers, radius, strict=True)
    need = math.ceil(samples.shape[0] / (8 * centers.shape[0]))
    return counts, int((counts >= need).sum().item())

### Probes

The two boundary samples sit at distance exactly $0.5$: $(1.5,1.0)-(1,1)=(0.5,0)$ and $(-1.0,0.5)-(-1,1)=(0,-0.5)$, both representable exactly in float32, so their squared distance is exactly $0.25=0.5^2$.

In [ ]:
counts1, covered1 = mode_coverage(S, CENTERS, 0.5)
counts_strict, _ = mode_coverage_strict(S, CENTERS, 0.5)

tie_counts, tie_covered = mode_coverage(torch.tensor([[0.5, 0.0]]), torch.tensor([[0.0, 0.0], [1.0, 0.0]]), 1.0)

counts3, covered3 = mode_coverage(S3, CENTERS, 0.5)
threshold3 = math.ceil(S3.shape[0] / (8 * CENTERS.shape[0]))

perm = torch.randperm(10, generator=torch.Generator().manual_seed(SEED))
counts_perm, covered_perm = mode_coverage(S[perm], CENTERS, 0.5)

invalid_inputs = [
    (torch.zeros(3), CENTERS, 0.5),                 # samples not 2-D
    (S, torch.zeros(4), 0.5),                       # centers not 2-D
    (torch.zeros(3, 3), CENTERS, 0.5),              # last dimensions differ
    (torch.zeros(0, 2), CENTERS, 0.5),              # n = 0
    (S, CENTERS, 0.0),                              # radius == 0
    (S, CENTERS, -1.0),                             # radius < 0
]
raised = []
for args in invalid_inputs:
    try:
        mode_coverage(*args)
    except ValueError:
        raised.append(True)
    else:
        raised.append(False)

print("probe 1:", counts1.tolist(), covered1, " strict:", counts_strict.tolist())
print("probe 3:", tie_counts.tolist(), tie_covered)
print("probe 4:", counts3.tolist(), threshold3, covered3)
print("probe 5:", counts_perm.tolist())
print("probe 6:", raised)

### What the count vector reveals

The count vector shows **how** the samples are distributed: a vector such as `[200, 16, 16, 0]` and a balanced `[60, 60, 60, 0]` both report three covered modes, but only the counts reveal that one mode dominates (collapse in progress) or that most samples land outside every radius (imprecision).
A single "number of covered modes" is a thresholded summary that hides the imbalance between modes and the fraction of samples counted nowhere, both of which change long before the covered number does.

### Answer check

In [ ]:
# Probe 1
assert counts1.tolist() == [2, 3, 1, 1] and covered1 == 4
# Probe 2
assert counts_strict.tolist() == [1, 2, 1, 1]
assert not torch.equal(counts_strict, counts1)
# Probe 3
assert tie_counts.tolist() == [1, 0] and tie_covered == 1
# Probe 4
assert S3.shape[0] == 64 and threshold3 == 2
assert counts3.tolist() == [20, 2, 1, 0] and covered3 == 2
# Probe 5
assert torch.equal(counts_perm, counts1) and covered_perm == covered1
# Probe 6
assert raised == [True] * len(invalid_inputs)
# dtype / shape / type
for counts, covered in ((counts1, covered1), (tie_counts, tie_covered), (counts3, covered3)):
    assert counts.dtype == torch.int64
    assert type(covered) is int
assert tuple(counts1.shape) == (4,) and tuple(tie_counts.shape) == (2,)